# Construction de la table de transactions enrichie **2014-2026** — table HYBRIDE

**But.** Produire la table complète des transactions du Congrès US (House + Sénat) **2014-2026** qui alimente la
phase de recherche/backtest. La table est **HYBRIDE** :
- **2020-2026 = le golden** (le travail « partie 1 » : sources officielles + OCR papier + non-coté, 90 487 txns) — réutilisé tel quel ;
- **2014-2019 = Quiver** (reconstruit depuis les sources, ~48 070 txns) — l'extension vers le passé.
- **Commissions recalculées point-in-time sur TOUTE la période** (par Congrès), y compris sur le golden (qui était
  sur le snapshot *actuel*). Enrichissement : secteur GICS + ETF proxy, parti/État/district/ancienneté.

> Pourquoi hybride : Quiver **sous-compte 2020-2026** (~65 k vs 90 k golden) car il ne suit pas le non-coté ni tout
> le papier OCR. On garde donc le golden pour 2020-26 et on n'utilise Quiver que pour 2014-19.
> **Limite assumée** : 2014-2019 = Quiver seul (pas de papier OCR ni non-coté) — à compléter plus tard avec la
> méthodo OCR de la partie 1.

**Sources (la portion Quiver 2014-2019 est reconstruite en direct, inlinée — aucun import `common/house/senate`) :**
- Transactions → **API Quiver** (`bulk/congresstrading`).
- Identité point-in-time → **legislators-current/historical.json**, champ `terms[]`.
- Commissions point-in-time → **historique git** de `committee-membership-current.yaml` (un snapshot par Congrès 113→119).
- Secteur GICS → **yfinance** (primaire) + repli **LLM** (Anthropic).
- **2020-2026** → lecture du **golden FINAL** du dépôt (réutilisation assumée du travail partie 1).

Caches propres dans `build_cache/` (re-runs rapides). Sortie : `table_congres_2014_2026.csv` (hybride).

> Dépendances : `pandas, numpy, requests, pyyaml, yfinance`, et `anthropic` (optionnel, repli secteur).

## 0 — Imports, chemins, clés API

In [1]:
import os, re, json, time, logging
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import requests

# Silence le logger yfinance : les tickers introuvables (HTTP 404) sont ATTENDUS (sociétés délistées →
# repli LLM), pas des erreurs — on ne veut pas en inonder la sortie du notebook.
for _noisy in ('yfinance', 'urllib3'):
    logging.getLogger(_noisy).setLevel(logging.CRITICAL)

# Localisation autonome du dossier S3S4 (où écrire) et du repo (pour .env).
def _find_paths():
    cands = [Path.cwd(), *Path.cwd().parents,
             Path.home()/'Downloads'/'Jupiter', Path.home()/'Downloads'/'0. Jupiter']
    for c in cands:
        s3 = c/'00. S3S4 en cours'
        if (c/'data'/'house'/'tables').exists() and s3.exists():
            return c, s3
    # repli : si on est déjà dans le dossier S3S4
    for c in [Path.cwd(), *Path.cwd().parents]:
        if c.name.startswith('00.') and c.name.endswith('en cours'):
            return c.parent, c
    raise RuntimeError('repo / dossier S3S4 introuvable')

REPO, S3 = _find_paths()
OUT   = S3/'table_congres_2014_2026.csv'
CACHE = S3/'build_cache'; CACHE.mkdir(exist_ok=True)

# .env (clés API) — lecture seule, depuis le repo.
for base in [REPO, S3, Path.home()/'Downloads'/'Jupiter']:
    f = base/'.env'
    if f.exists():
        for line in f.read_text().splitlines():
            if '=' in line and not line.strip().startswith('#'):
                k, v = line.split('=', 1)
                os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
        break

QUIVER_KEY = os.environ.get('QUIVER_API_KEY')
HAS_LLM    = bool(os.environ.get('ANTHROPIC_API_KEY'))
S = requests.Session()
S.headers.update({'User-Agent': 'congress-trading-research/1.0 (poli, sans evasion)'})

print('REPO :', REPO)
print('S3   :', S3)
print('OUT  :', OUT)
print('Quiver key :', 'OK' if QUIVER_KEY else 'MANQUANTE', '| LLM repli :', HAS_LLM)

REPO : /Users/lemairealice/Downloads/Jupiter
S3   : /Users/lemairealice/Downloads/Jupiter/00. S3S4 en cours
OUT  : /Users/lemairealice/Downloads/Jupiter/00. S3S4 en cours/table_congres_2014_2026.csv
Quiver key : OK | LLM repli : True


### Les deux caches du dossier (à ne pas confondre)

- **`build_cache/`** — *caches de CE notebook* : Quiver brut (`quiver_raw.csv`), référentiels legislators
  (`leg_*.json`), snapshots de commissions par Congrès (`committees/`), et `ticker_sector.json`. **Régénérables**,
  gitignorés. Ils servent uniquement à rendre les re-runs rapides — le 1er run les reconstruit depuis les sources.
- **`cache/`** — *cache du notebook de RECHERCHE* (`Congress_Trading_Signal_Recherche.ipynb`) : cours yfinance
  (`prices/`) + facteurs Fama-French (`ff_factors.csv`) pour le backtest. **Rien à voir avec ce notebook-ci.**

Aucun des deux n'est lu par l'autre notebook. Tous deux gitignorés (régénérables).

## 1 — Journal des transactions (API Quiver, live)

On récupère le dump `bulk/congresstrading` (House + Sénat, ~114 k lignes, 2012-2026), on le met en cache brut, puis
on normalise **exactement** comme le notebook de recherche : opération `Purchase→buy`…, ticker coté propre
(`BRK.B→BRK-B`), `size_usd` = borne basse de la fourchette STOCK Act. On garde les lignes `filed`/`traded` valides
et `filed.year ∈ [2014, 2026]`. `party_quiver` = le parti tel que Quiver le donne (parti *courant*) — gardé pour
audit ; le parti **point-in-time** est reconstruit plus bas.

In [2]:
QUIVER_URL  = 'https://api.quiverquant.com/beta/bulk/congresstrading'
MIN_BRACKET = 1001.0

def _norm_op(x):
    s = str(x).lower()
    return 'buy' if 'purchase' in s else 'sell' if 'sale' in s else 'exch' if 'exchange' in s else 'other'

def _norm_ticker(t):
    if not isinstance(t, str):
        return None
    s = t.strip().upper()
    if re.fullmatch(r'[A-Z]{1,5}', s):
        return s
    if re.fullmatch(r'[A-Z]{1,4}\.[A-Z]', s):
        return s.replace('.', '-')
    return None

def fetch_quiver_live(force=False):
    raw = CACHE/'quiver_raw.csv'
    if raw.exists() and not force:
        return pd.read_csv(raw, dtype=str)
    if not QUIVER_KEY:
        raise RuntimeError('QUIVER_API_KEY absente du .env')
    last = None
    for attempt in range(4):
        try:
            r = S.get(QUIVER_URL, headers={'Authorization': f'Bearer {QUIVER_KEY}'}, timeout=180)
            r.raise_for_status()
            q = pd.DataFrame(r.json())
            q.to_csv(raw, index=False)
            return q
        except requests.exceptions.RequestException as e:
            last = e
            time.sleep(3 * (attempt + 1))
    raise RuntimeError(f'échec fetch Quiver : {last}')

q_raw = fetch_quiver_live()
print('Quiver brut :', q_raw.shape, '| chambres :', q_raw['Chamber'].value_counts().to_dict())

Quiver brut : (113682, 20) | chambres : {'Representatives': 100340, 'Senate': 13342}


In [3]:
def normalize_quiver(q):
    ch = q['Chamber'].astype(str).str.lower()
    chamber = np.where(ch.str.contains('rep'), 'house',
              np.where(ch.str.contains('sen'), 'senate', None))
    j = pd.DataFrame({
        'chamber'    : chamber,
        'bioguide'   : q['BioGuideID'],
        'name'       : q['Name'],
        'ticker'     : q['Ticker'].map(_norm_ticker),
        'ticker_type': q.get('TickerType'),
        'op'         : q['Transaction'].map(_norm_op),
        'traded'     : pd.to_datetime(q['Traded'], errors='coerce'),
        'filed'      : pd.to_datetime(q['Filed'],  errors='coerce'),
        'size_usd'   : pd.to_numeric(q['Trade_Size_USD'], errors='coerce').clip(lower=MIN_BRACKET),
        'party_quiver': q['Party'],
    })
    j = j[j['filed'].notna() & j['traded'].notna()]
    j = j[(j['filed'].dt.year >= 2014) & (j['filed'].dt.year <= 2026)].reset_index(drop=True)
    return j

journal = normalize_quiver(q_raw)
ASOF = journal['traded'].fillna(journal['filed'])   # date de référence point-in-time
print('journal 2014-2026 :', journal.shape, '| par chambre :', journal['chamber'].value_counts().to_dict())
print('par année (filed) :', journal['filed'].dt.year.value_counts().sort_index().to_dict())

journal 2014-2026 : (113682, 10) | par chambre : {'house': 100340, 'senate': 13342}
par année (filed) : {2014: 5099, 2015: 5360, 2016: 7147, 2017: 8821, 2018: 11140, 2019: 10503, 2020: 12392, 2021: 7656, 2022: 10095, 2023: 9715, 2024: 7366, 2025: 13151, 2026: 5237}


## 2 — Identité **point-in-time** (parti / État / district / ancienneté)

Source : `legislators-current.json` + `legislators-historical.json`. Chaque législateur a une liste de mandats
`terms[]` (chacun avec `start, end, party, state, district, type`). Pour chaque transaction, on prend le mandat
**actif à la date du trade** (repli : mandat le plus proche). `years_in_office` = année du trade − année du 1er
mandat. `party` (point-in-time) = parti du mandat actif (« Democrat/Republican/Independent », comme le golden) ;
`party_quiver` reste à part.

In [4]:
LEG_CUR  = 'https://unitedstates.github.io/congress-legislators/legislators-current.json'
LEG_HIST = 'https://unitedstates.github.io/congress-legislators/legislators-historical.json'

def _fetch_json(url, fname):
    f = CACHE/fname
    if f.exists():
        return json.loads(f.read_text())
    j = S.get(url, timeout=120).json()
    f.write_text(json.dumps(j))
    return j

def build_term_index():
    people = _fetch_json(LEG_CUR, 'leg_current.json') + _fetch_json(LEG_HIST, 'leg_historical.json')
    idx, first = {}, {}
    for p in people:
        bio = (p.get('id') or {}).get('bioguide')
        if not bio:
            continue
        terms = []
        for t in (p.get('terms') or []):
            st = t.get('start')
            if not st:
                continue
            en = t.get('end')
            terms.append((pd.Timestamp(st),
                          pd.Timestamp(en) if en else pd.Timestamp('2100-01-01'),
                          t.get('party'), t.get('state'), t.get('district'),
                          'house' if t.get('type') == 'rep' else 'senate'))
        if not terms:
            continue
        terms.sort()
        idx[bio] = terms
        first[bio] = min(t[0].year for t in terms)
    return idx, first

TERM_IDX, FIRST_YEAR = build_term_index()
print('législateurs indexés :', len(TERM_IDX))

def term_at(bio, date):
    terms = TERM_IDX.get(bio)
    if not terms or pd.isna(date):
        return (None, None, None, None)
    for st, en, party, state, dist, ch in terms:
        if st <= date <= en:
            return (party, state, dist, ch)
    before = [t for t in terms if t[0] <= date]
    c = before[-1] if before else terms[0]
    return (c[2], c[3], c[4], c[5])

_pit = [term_at(b, d) for b, d in zip(journal['bioguide'], ASOF)]
journal['party']    = [x[0] for x in _pit]
journal['state']    = [x[1] for x in _pit]
journal['district'] = [x[2] for x in _pit]
journal['years_in_office'] = [
    max(0, int(d.year) - FIRST_YEAR[b]) if (b in FIRST_YEAR and pd.notna(d)) else None
    for b, d in zip(journal['bioguide'], ASOF)]
print('parti point-in-time résolu :', journal['party'].notna().mean().round(3),
      '| ancienneté résolue :', journal['years_in_office'].notna().mean().round(3))

législateurs indexés : 12767
parti point-in-time résolu : 1.0 | ancienneté résolue : 1.0


## 3 — Commissions **point-in-time** (un snapshot par Congrès)

Le référentiel `committee-membership-current.yaml` ne contient que le Congrès *courant*. Mais son **historique git**
EST la série de snapshots : à chaque nouveau Congrès le fichier est réécrit. On récupère donc, via l'API GitHub, le
commit représentatif (mi-Congrès) pour chaque Congrès **113→119**, et le YAML brut à ce SHA — avec le
`committees-current.yaml` du **même SHA** pour résoudre `thomas_id → nom de commission` d'époque. La transaction
hérite de la composition du Congrès actif à sa date. **Repli explicite** : si un snapshot manque, on met `NaN`
(jamais rétro-remplir avec l'actuel).

In [5]:
import yaml
GH  = 'https://api.github.com/repos/unitedstates/congress-legislators/commits'
RAW = 'https://raw.githubusercontent.com/unitedstates/congress-legislators'
# date représentative (mi-Congrès) pour chaque Congrès
SNAP_DATE = {113:'2014-06-01', 114:'2016-06-01', 115:'2018-06-01', 116:'2020-06-01',
             117:'2022-06-01', 118:'2024-06-01', 119:'2026-06-01'}
KEY_PATTERNS = ('Financial Services', 'Committee on Finance', 'Ways and Means',
                'Banking', 'Armed Services', 'Intelligence')

def congress_of(year):
    return 113 + (int(year) - 2013) // 2

def _commit_before(path, until):
    r = S.get(GH, params={'path': path, 'until': until + 'T00:00:00Z', 'per_page': 1}, timeout=30)
    r.raise_for_status()
    j = r.json()
    return j[0]['sha'] if j else None

def load_committee_snapshot(congress):
    d = CACHE/'committees'/str(congress); d.mkdir(parents=True, exist_ok=True)
    memf, comf = d/'membership.yaml', d/'committees.yaml'
    if not (memf.exists() and comf.exists()):
        until = SNAP_DATE[congress]
        sha = _commit_before('committee-membership-current.yaml', until)
        if not sha:
            return None
        memf.write_text(S.get(f'{RAW}/{sha}/committee-membership-current.yaml', timeout=60).text)
        comf.write_text(S.get(f'{RAW}/{sha}/committees-current.yaml', timeout=60).text)
    mem = yaml.safe_load(memf.read_text())
    com = yaml.safe_load(comf.read_text())
    code_to_name = {c['thomas_id']: c['name'] for c in com if 'thomas_id' in c}
    bio2c = defaultdict(set)
    for code, members in mem.items():
        cname = code_to_name.get(code, code)
        for m in members:
            if m.get('bioguide'):
                bio2c[m['bioguide']].add(cname)
    return {b: '; '.join(sorted(cs)) for b, cs in bio2c.items()}

SNAPS = {}
for c in range(113, 120):
    SNAPS[c] = load_committee_snapshot(c)
    n = len(SNAPS[c]) if SNAPS[c] else 0
    print(f'  Congrès {c} ({SNAP_DATE[c][:4]}) : {n} membres avec commissions')

def committees_at(bio, year):
    snap = SNAPS.get(congress_of(year)) if pd.notna(year) else None
    return snap.get(bio) if snap else None

def key_flag(memb):
    if not isinstance(memb, str):
        return pd.NA
    return any(p in memb for p in KEY_PATTERNS)

journal['congress'] = [congress_of(d.year) if pd.notna(d) else None for d in ASOF]
journal['committee_membership'] = [committees_at(b, d.year if pd.notna(d) else None)
                                   for b, d in zip(journal['bioguide'], ASOF)]
journal['committees_key_flag'] = [key_flag(m) for m in journal['committee_membership']]
print('commissions résolues :', journal['committee_membership'].notna().mean().round(3))

  Congrès 113 (2014) : 531 membres avec commissions
  Congrès 114 (2016) : 536 membres avec commissions
  Congrès 115 (2018) : 528 membres avec commissions
  Congrès 116 (2020) : 529 membres avec commissions
  Congrès 117 (2022) : 526 membres avec commissions
  Congrès 118 (2024) : 529 membres avec commissions
  Congrès 119 (2026) : 528 membres avec commissions
commissions résolues : 0.99


## 4 — Secteur **GICS + ETF proxy** (yfinance primaire + repli LLM)

Logique reprise du pipeline (inlinée) : yfinance (`info['sector']`) → map vers les 11 secteurs GICS, puis repli LLM
(Anthropic, par lots) pour les tickers que yfinance ne résout pas. Le prompt LLM reconnaît explicitement les
**sociétés délistées / renommées / rachetées** (ex. TWTR, ANTM, TWX) et leur donne quand même un secteur ; une passe
`retry_none` ré-attaque les tickers restés `none`. Restent légitimement `none` : non-coté (obligations, munis,
Treasury, privé) **et** ETF/fonds **diversifiés ou obligataires** (VOO, SCHD, JEPI, TIP…), qui n'ont pas de secteur
GICS unique — comme SPY/QQQ. `etf_proxy` = ETF SPDR Select Sector du secteur. Cache propre du notebook :
`build_cache/ticker_sector.json` (re-run instantané).

> ⚠️ 1er run : yfinance sur ~5 k tickers est **le poste lent** (~30-60 min, throttling Yahoo possible). Le cache
> rend les re-runs immédiats. Le bruit « HTTP Error 404 » de yfinance est **silencé** (logger en CRITICAL) : ces
> 404 sont attendus (tickers délistés) et basculent au repli LLM, ce ne sont pas des erreurs. Sans
> `ANTHROPIC_API_KEY`, le repli LLM est sauté (les non-résolus → `none`).

In [6]:
GICS_TO_ETF = {'Energy':'XLE', 'Materials':'XLB', 'Industrials':'XLI', 'Consumer Discretionary':'XLY',
    'Consumer Staples':'XLP', 'Health Care':'XLV', 'Financials':'XLF', 'Information Technology':'XLK',
    'Communication Services':'XLC', 'Utilities':'XLU', 'Real Estate':'XLRE'}
GICS_SECTORS = tuple(GICS_TO_ETF.keys())
YF_SECTOR_TO_GICS = {'Technology':'Information Technology', 'Financial Services':'Financials',
    'Healthcare':'Health Care', 'Consumer Cyclical':'Consumer Discretionary',
    'Consumer Defensive':'Consumer Staples', 'Energy':'Energy', 'Industrials':'Industrials',
    'Basic Materials':'Materials', 'Real Estate':'Real Estate', 'Utilities':'Utilities',
    'Communication Services':'Communication Services'}
MANUAL_OVERRIDES = {'FNA':'Health Care', 'SMCYY':'Communication Services', 'LRN':'Consumer Discretionary',
    'SHLD':'Industrials', 'HURA':'Materials', 'URNM':'Materials',
    'SLYG':None, 'TNA':None, 'SPY':None, 'QQQ':None, 'IVW':None, 'PDBC':None, 'BITB':None}
_VALID = re.compile(r'^[A-Z][A-Z.\-]{0,6}$')

def _yf_one(t):
    import yfinance as yf
    try:
        raw = (yf.Ticker(t).info or {}).get('sector')
        return YF_SECTOR_TO_GICS.get(raw) if raw else None
    except Exception:
        return None

_SECTOR_TOOL = {'name':'map_sectors', 'description':'Renvoie le secteur GICS de chaque ticker US fourni.',
  'input_schema':{'type':'object','properties':{'mappings':{'type':'array','items':{'type':'object','properties':{
      'ticker':{'type':'string'},
      'sector_gics':{'type':['string','null'],'enum':[*GICS_SECTORS, None]},
      'is_listed':{'type':'boolean'}}, 'required':['ticker','sector_gics','is_listed']}}}, 'required':['mappings']}}
_SECTOR_PROMPT = ('Tu reçois des tickers boursiers US (déclarations de transactions du Congrès, 2014-2026, donc '
  'BEAUCOUP de tickers anciens, délistés, renommés ou rachetés). Pour CHAQUE ticker, donne le secteur GICS de '
  'l\'émetteur, STRICTEMENT parmi : ' + ', '.join(GICS_SECTORS) + '. '
  'IMPORTANT : si le ticker correspond à une société cotée BIEN CONNUE, donne son secteur MÊME si elle est '
  'aujourd\'hui délistée / renommée / rachetée (ex. TWTR=Communication Services, TWX=Communication Services, '
  'ANTM=Health Care, KSU=Industrials, LLL=Industrials, PRAH=Health Care, LTD=Consumer Discretionary). '
  'Mets sector_gics=null UNIQUEMENT si : (a) ce n\'est pas une action (obligation, bon du Trésor, muni, fonds non '
  'coté, titre privé), OU (b) c\'est un ETF/fonds DIVERSIFIÉ ou OBLIGATAIRE sans secteur GICS dominant '
  '(ex. SPY, QQQ, VOO, SCHD, JEPI, TIP, AGG). Pour un ETF SECTORIEL, donne le secteur dominant. Ne devine pas '
  'au hasard pour un ticker inconnu. Recopie ticker à l\'identique.\n\nTickers :\n{tickers}')

def resolve_llm(tickers, batch=40):
    if not HAS_LLM or not tickers:
        return {}
    import anthropic
    cli = anthropic.Anthropic()
    out = {}
    for i in range(0, len(tickers), batch):
        chunk = tickers[i:i+batch]
        listing = '\n'.join(f'- {t}' for t in chunk)
        for attempt in range(4):
            try:
                resp = cli.messages.create(model='claude-sonnet-4-6', max_tokens=8000, tools=[_SECTOR_TOOL],
                    tool_choice={'type':'tool','name':'map_sectors'},
                    messages=[{'role':'user','content':_SECTOR_PROMPT.format(tickers=listing)}])
                for blk in resp.content:
                    if getattr(blk,'type',None)=='tool_use' and blk.name=='map_sectors':
                        for m in blk.input.get('mappings', []):
                            tk=(m.get('ticker') or '').upper().strip(); sec=m.get('sector_gics')
                            # On fait confiance à la décision de secteur du LLM (le prompt impose null pour le
                            # non-coté et les ETF diversifiés/obligataires) — sans gate is_listed, qui rejetait
                            # à tort les sociétés délistées mais connues (TWTR, ANTM…).
                            out[tk]= sec if sec in GICS_TO_ETF else None
                break
            except Exception:
                time.sleep(min(2**attempt, 20))
        for t in chunk:
            out.setdefault(t, None)
    return out

def resolve_sectors(tickers, with_llm=True, checkpoint=200, pause=0.1, retry_none=True):
    cachef = CACHE/'ticker_sector.json'
    cache = json.loads(cachef.read_text()) if cachef.exists() else {}
    tickers = sorted({t for t in tickers if isinstance(t, str) and _VALID.match(t)})
    todo = [t for t in tickers if t not in cache]
    print(f'  tickers : {len(tickers)} distincts | à résoudre : {len(todo)} (cache : {len(tickers)-len(todo)})')
    # Phase 1 — yfinance (factuel), avec checkpoint régulier du cache (résumable si interrompu)
    for n, t in enumerate(todo, 1):
        g = _yf_one(t)
        cache[t] = {'yf': g, 'llm': None, 'sector_gics': g, 'source': 'yfinance' if g else 'none'}
        if pause:
            time.sleep(pause)
        if n % checkpoint == 0:
            cachef.write_text(json.dumps(cache, ensure_ascii=False, indent=1))
            print(f'    yfinance {n}/{len(todo)} (checkpoint)')
    cachef.write_text(json.dumps(cache, ensure_ascii=False, indent=1))
    # Phase 2 — repli LLM sur les tickers que yfinance n'a pas résolus
    targets = [t for t in todo if cache[t]['source'] == 'none']
    print(f'  yfinance: {len(todo)-len(targets)}/{len(todo)} résolus | repli LLM sur {len(targets)}')
    if with_llm and targets:
        llm_res = resolve_llm(targets, batch=40)
        for t in targets:
            g = llm_res.get(t)
            if g:
                cache[t].update({'llm': g, 'sector_gics': g, 'source': 'llm'})
        cachef.write_text(json.dumps(cache, ensure_ascii=False, indent=1))
    # Phase 3 — RE-attaque LLM (une seule fois, marquée) les tickers en cache 'none' : récupère les sociétés
    # délistées/renommées bien connues que la 1re passe ratait. ETF diversifiés/obligataires restent 'none'.
    if retry_none and with_llm:
        stale = [t for t in tickers
                 if cache.get(t, {}).get('source') == 'none' and not cache.get(t, {}).get('llm_retry')]
        if stale:
            print(f'  retry_none : ré-attaque LLM sur {len(stale)} tickers non résolus')
            llm2 = resolve_llm(stale, batch=40)
            recovered = 0
            for t in stale:
                cache[t]['llm_retry'] = True
                g = llm2.get(t)
                if g:
                    cache[t].update({'llm': g, 'sector_gics': g, 'source': 'llm'})
                    recovered += 1
            print(f'  retry_none : {recovered} récupérés')
            cachef.write_text(json.dumps(cache, ensure_ascii=False, indent=1))
    return cache

SEC = resolve_sectors(journal['ticker'].dropna().unique().tolist(), with_llm=HAS_LLM)

def _gics(t):
    return (SEC.get(t) or {}).get('sector_gics') if isinstance(t, str) else None
def _src(t):
    return (SEC.get(t) or {}).get('source', 'none') if isinstance(t, str) else 'none'

journal['sector_gics']   = journal['ticker'].map(_gics)
journal['etf_proxy']     = journal['sector_gics'].map(lambda s: GICS_TO_ETF.get(s) if s else None)
journal['sector_source'] = journal['ticker'].map(_src)
for tk, sec in MANUAL_OVERRIDES.items():
    m = journal['ticker'] == tk
    if m.any():
        journal.loc[m, 'sector_gics']   = sec
        journal.loc[m, 'etf_proxy']     = GICS_TO_ETF.get(sec) if sec else None
        journal.loc[m, 'sector_source'] = 'manual'
_msk = journal['ticker'].notna()
print('secteur résolu (lignes avec ticker) :', journal.loc[_msk, 'sector_gics'].notna().mean().round(3))
print('  par source :', journal.loc[_msk, 'sector_source'].value_counts(dropna=False).to_dict())

  tickers : 4848 distincts | à résoudre : 0 (cache : 4848)
  yfinance: 0/0 résolus | repli LLM sur 0
secteur résolu (lignes avec ticker) : 0.975
  par source : {'yfinance': 90299, 'llm': 19291, 'none': 2563, 'manual': 237}


## 5 — Table Quiver (référence) puis **assemblage HYBRIDE**

On assemble d'abord la table **Quiver pure** 2014-2026 (sert de référence/validation + fournit la portion 2014-2019),
écrite dans `build_cache/`. Puis on construit le **livrable HYBRIDE** : **ton golden 2020-2026** (travail complet,
papier OCR + non-coté) **+ Quiver 2014-2019**. Les commissions sont **recalculées point-in-time** aussi sur le golden
(le golden était sur le snapshot *actuel*). Frontière propre (Quiver `filed ≤ 2019`, golden `disclosure ≥ 2020`) →
pas de double-comptage.

In [7]:
journal['bioguide_id'] = journal['bioguide']   # alias attendu par le notebook de recherche
COLS = ['chamber', 'bioguide', 'bioguide_id', 'name', 'party', 'state', 'district',
        'committee_membership', 'committees_key_flag', 'years_in_office',
        'ticker', 'ticker_type', 'op', 'traded', 'filed', 'size_usd',
        'sector_gics', 'etf_proxy', 'sector_source', 'party_quiver', 'congress']
table = journal[COLS].copy()
QUIVER_REF = CACHE/'quiver_table_2014_2026.csv'
table.to_csv(QUIVER_REF, index=False)   # référence Quiver pure (validation) — PAS le livrable
print('table Quiver (référence) :', table.shape, '->', QUIVER_REF.name)

table Quiver (référence) : (113682, 21) -> quiver_table_2014_2026.csv


In [8]:
import glob
# 1) Golden FINAL 2020-2026 (ton travail : officiel + OCR + non-coté) — réutilisation VOULUE
gfiles = (glob.glob(str(REPO/'data/house/tables/*/06_house_*_FINAL.csv')) +
          glob.glob(str(REPO/'data/senate/tables/*/06_senate_*_FINAL.csv')))
golden = pd.concat([pd.read_csv(f, dtype=str) for f in gfiles], ignore_index=True)
gdisc = pd.to_datetime(golden['disclosure_date'], errors='coerce')
golden = golden[gdisc.dt.year >= 2020].reset_index(drop=True)   # garde-fou anti-chevauchement
gtxn_year = (pd.to_datetime(golden['transaction_date'], errors='coerce')
             .fillna(pd.to_datetime(golden['disclosure_date'], errors='coerce')).dt.year)

# 2) Commissions POINT-IN-TIME recalculées sur le golden (remplace le snapshot 'actuel')
golden_com = [committees_at(b, y) for b, y in zip(golden['bioguide_id'], gtxn_year)]

def _amount_low(s):   # borne basse de "$50,001 - $100,000" -> 50001.0 (clip 1001), pour matcher Quiver
    if not isinstance(s, str):
        return None
    m = re.findall(r'[\d,]+', s)
    if not m:
        return None
    try:
        return max(1001.0, float(m[0].replace(',', '')))
    except Exception:
        return None

# 3) Harmoniser le golden au schéma unifié (mêmes noms que la table Quiver)
gh = pd.DataFrame({
    'chamber': golden['chamber'], 'bioguide': golden['bioguide_id'], 'bioguide_id': golden['bioguide_id'],
    'name': golden['declarant_name'], 'party': golden['party'],
    'state': golden.get('state_district'), 'district': None,
    'committee_membership': golden_com, 'committees_key_flag': [key_flag(m) for m in golden_com],
    'years_in_office': golden.get('years_in_office'),
    'ticker': golden['ticker'], 'ticker_type': golden.get('asset_type'),
    'op': golden['operation_type'].map(_norm_op),
    'traded': pd.to_datetime(golden['transaction_date'], errors='coerce'),
    'filed': pd.to_datetime(golden['disclosure_date'], errors='coerce'),
    'size_usd': golden['amount_range'].map(_amount_low),
    'sector_gics': golden['sector_gics'], 'etf_proxy': golden.get('etf_proxy'),
    'sector_source': golden.get('sector_source'), 'party_quiver': None,
    'congress': [congress_of(y) if pd.notna(y) else None for y in gtxn_year],
})
gh['source'] = 'golden-2020-2026'

# 4) Quiver 2014-2019 (la portion extension)
q = table.copy(); q['source'] = 'quiver-2014-2019'
q1419 = q[pd.to_datetime(q['filed']).dt.year <= 2019].reset_index(drop=True)

# 5) Concat -> HYBRIDE (le livrable)
hybride = pd.concat([q1419, gh[q1419.columns]], ignore_index=True)

# size_usd : défaut borne basse 1001 pour les ~0,6 % de lignes golden au montant OCR non capté
# (sinon NaN contaminerait les variantes pondérées par taille du backtest).
hybride['size_usd'] = pd.to_numeric(hybride['size_usd'], errors='coerce').fillna(1001.0).clip(lower=1001.0)

# Normaliser les tickers au format du panel de prix (BRK.B -> BRK-B ; accepte déjà-tiret ; idempotent côté Quiver)
def _norm_ticker_panel(t):
    if not isinstance(t, str):
        return None
    s = t.strip().upper()
    if re.fullmatch(r'[A-Z]{1,5}', s):
        return s
    if re.fullmatch(r'[A-Z]{1,4}[.\-][A-Z]', s):
        return s.replace('.', '-')
    return None
hybride['ticker'] = hybride['ticker'].map(_norm_ticker_panel)

hybride.to_csv(OUT, index=False)
print('golden 2020-2026 :', len(gh), '| Quiver 2014-2019 :', len(q1419), '| HYBRIDE :', len(hybride))
print('écrit :', OUT, '| par source :', hybride['source'].value_counts().to_dict())
hybride.head(4)

golden 2020-2026 : 90487 | Quiver 2014-2019 : 48070 | HYBRIDE : 138557
écrit : /Users/lemairealice/Downloads/Jupiter/00. S3S4 en cours/table_congres_2014_2026.csv | par source : {'golden-2020-2026': 90487, 'quiver-2014-2019': 48070}


,chamber,bioguide,bioguide_id,name,party,state,district,committee_membership,committees_key_flag,years_in_office,...,op,traded,filed,size_usd,sector_gics,etf_proxy,sector_source,party_quiver,congress,source
0,senate,G000584,G000584,Greg Gianforte,Republican,MT,0.0,HSIF14; HSIF16; HSIF17; House Committee on Ene...,False,2,...,sell,2019-12-24,2019-02-02,100001.0,Consumer Discretionary,XLY,yfinance,Republican,116,quiver-2014-2019
1,house,G000563,G000563,Bob Gibbs,Republican,OH,7.0,HSGO28; HSPW07; HSPW12; House Committee on Ove...,False,8,...,buy,2019-12-23,2019-12-23,1001.0,Communication Services,XLC,yfinance,Republican,116,quiver-2014-2019
2,house,M001193,M001193,Thomas Macarthur,Republican,NJ,3.0,NaN,<NA>,4,...,buy,2019-12-21,2019-01-11,1001.0,Financials,XLF,yfinance,Republican,116,quiver-2014-2019
3,senate,G000584,G000584,Greg Gianforte,Republican,MT,0.0,HSIF14; HSIF16; HSIF17; House Committee on Ene...,False,2,...,buy,2019-12-21,2019-01-23,15001.0,Financials,XLF,yfinance,Republican,116,quiver-2014-2019


## 6 — Vérifications (table HYBRIDE)

(0) composition par source (golden 2020-26 préservé = 90 487) ; (1) comptes par année ; (2) couverture secteur ;
(3) couverture commissions **point-in-time** par année ; (4) **concordance** de la méthode Quiver vs golden sur
2020-2026 (la table Quiver de référence, pour prouver que l'enrichissement Quiver utilisé en 2014-2019 « colle »).

In [9]:
print('=== (0) composition HYBRIDE par source ===')
print('  ', hybride['source'].value_counts().to_dict())
print('  golden 2020-26 préservé :', int((hybride['source']=='golden-2020-2026').sum()), '(= ton golden, 0 perte)')
print('  Quiver 2014-19 ajouté   :', int((hybride['source']=='quiver-2014-2019').sum()))

print('\n=== (1) lignes par année (filed/disclosure) ===')
print(pd.to_datetime(hybride['filed'], errors='coerce').dt.year.value_counts().sort_index())

print('\n=== (2) couverture secteur (lignes avec ticker) ===')
tk = hybride[hybride['ticker'].notna()]
print('  sector_gics non-null :', round(tk['sector_gics'].notna().mean(), 3))
print('  par source secteur :', hybride['sector_source'].value_counts(dropna=False).to_dict())

print('\n=== (3) couverture commissions par année (point-in-time partout) ===')
yr = pd.to_datetime(hybride['traded'], errors='coerce').dt.year
cov = hybride.assign(_ty=yr).groupby('_ty')['committee_membership'].apply(lambda s: round(s.notna().mean(), 3))
print(cov)

=== (0) composition HYBRIDE par source ===
   {'golden-2020-2026': 90487, 'quiver-2014-2019': 48070}
  golden 2020-26 préservé : 90487 (= ton golden, 0 perte)
  Quiver 2014-19 ajouté   : 48070

=== (1) lignes par année (filed/disclosure) ===
filed
2014     5099
2015     5360
2016     7147
2017     8821
2018    11140
2019    10503
2020    17638
2021    13652
2022    15602
2023    11649
2024    10001
2025    15065
2026     6880
Name: count, dtype: int64

=== (2) couverture secteur (lignes avec ticker) ===
  sector_gics non-null : 0.968
  par source secteur : {'yfinance': 101010, 'none': 18798, 'llm': 18423, 'manual': 326}

=== (3) couverture commissions par année (point-in-time partout) ===
_ty
2012.0    0.000
2013.0    0.998
2014.0    0.989
2015.0    0.997
2016.0    0.998
2017.0    0.982
2018.0    0.995
2019.0    0.999
2020.0    0.994
2021.0    0.992
2022.0    0.995
2023.0    0.999
2024.0    0.998
2025.0    0.967
2026.0    0.989
2202.0    0.000
2220.0    0.000
3031.0    0.000
Name: comm

In [10]:
# (4) Concordance overlap 2020-2026 vs FINAL golden (comparaison seule).
import glob
fin_files = (glob.glob(str(REPO/'data/house/tables/*/06_house_*_FINAL.csv')) +
             glob.glob(str(REPO/'data/senate/tables/*/06_senate_*_FINAL.csv')))
if fin_files:
    fin = pd.concat([pd.read_csv(f, dtype=str) for f in fin_files], ignore_index=True)
    # secteur par ticker : mode FINAL vs notre table
    fin_sec = fin.dropna(subset=['ticker','sector_gics']).groupby('ticker')['sector_gics'].agg(lambda s: s.value_counts().index[0])
    our_sec = table.dropna(subset=['ticker','sector_gics']).groupby('ticker')['sector_gics'].agg(lambda s: s.value_counts().index[0])
    common = sorted(set(fin_sec.index) & set(our_sec.index))
    agree = sum(fin_sec[t] == our_sec[t] for t in common)
    print(f'SECTEUR : {len(common)} tickers communs | accord {agree}/{len(common)} = {agree/max(1,len(common)):.1%}')
    diff = [(t, fin_sec[t], our_sec[t]) for t in common if fin_sec[t] != our_sec[t]]
    if diff:
        print('  divergences (ticker, FINAL, nous) :', diff[:15])

    # parti par bioguide : FINAL (parti courant) vs notre party point-in-time (≠ attendu pour transfuges)
    fin_party = fin.dropna(subset=['bioguide_id','party']).groupby('bioguide_id')['party'].agg(lambda s: s.value_counts().index[0])
    our_party = (table[pd.to_datetime(table['traded']).dt.year>=2020]
                 .dropna(subset=['bioguide_id','party']).groupby('bioguide_id')['party']
                 .agg(lambda s: s.value_counts().index[0]))
    cb = sorted(set(fin_party.index) & set(our_party.index))
    pa = sum(fin_party[b] == our_party[b] for b in cb)
    print(f'PARTI   : {len(cb)} membres communs | accord {pa}/{len(cb)} = {pa/max(1,len(cb)):.1%} (écarts = changements d\'affiliation)')

    # committees_key_flag : FINAL vs nous (overlap)
    if 'committees_key_flag' in fin.columns:
        fin_key = fin.dropna(subset=['bioguide_id']).assign(
            kf=fin['committees_key_flag'].astype(str).str.lower().isin(['true','1'])
        ).groupby('bioguide_id')['kf'].max()
        our_key = (table[pd.to_datetime(table['traded']).dt.year>=2020]
                   .dropna(subset=['bioguide_id']).assign(kf=table['committees_key_flag']==True)
                   .groupby('bioguide_id')['kf'].max())
        kb = sorted(set(fin_key.index) & set(our_key.index))
        ka = sum(bool(fin_key[b]) == bool(our_key[b]) for b in kb)
        print(f'KEY-FLAG: {len(kb)} membres communs | accord {ka}/{len(kb)} = {ka/max(1,len(kb)):.1%}')
else:
    print('FINAL introuvable — concordance sautée (la table reste valide).')

SECTEUR : 2840 tickers communs | accord 2791/2840 = 98.3%
  divergences (ticker, FINAL, nous) : [('AESE', 'Communication Services', 'Consumer Discretionary'), ('ASHGY', 'Materials', 'Consumer Staples'), ('ASHTY', 'Materials', 'Consumer Staples'), ('ATASY', 'Industrials', 'Information Technology'), ('BKI', 'Financials', 'Information Technology'), ('CCLAY', 'Materials', 'Industrials'), ('CCMP', 'Information Technology', 'Materials'), ('CLGX', 'Industrials', 'Real Estate'), ('CTRL', 'Industrials', 'Information Technology'), ('DHS', 'Consumer Discretionary', 'Financials'), ('DIA', 'Financials', 'Industrials'), ('DIAL', 'Communication Services', 'Financials'), ('DLS', 'Industrials', 'Financials'), ('DTD', 'Consumer Discretionary', 'Industrials'), ('DUSA', 'Energy', 'Information Technology')]
PARTI   : 256 membres communs | accord 256/256 = 100.0% (écarts = changements d'affiliation)
KEY-FLAG: 256 membres communs | accord 196/256 = 76.6%
